In [ ]:
import os, re
import select
import datetime

In [ ]:
FIFO_PATH = "/tmp/.dcmRxInfo.log"
EVENT_LOG = os.environ['MRI_SCANNER_LOG_DIR'] + "MrMeas_container.log"

In [ ]:
# Test with command:   date '+%Y-%m-%d  %H:%M:%S' > /tmp/.dcmRxInfo.log

In [ ]:
   def check_inline_export_log (scanner_events_dictionary,
                                export_log='/var/log/dcmRxInfo.log'):

      """
         This routine will parse the log output from the inline real-time export log.
         It will determine the start (MEAS_START) and stop (MEAS_FINISHED) of image
         reconstruction, as flags for these are not reliably written to the system's
         logs on the console that routines here get most of their info from.
      """

      event_date_time_00 = re.compile(r'\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}')

      with open (export_log, 'r') as fifo_rt:
         print (f"FIFO '{export_log}' opened for reading. Waiting for data...")

         while True:
            select.select([fifo_rt], [], [], 0.1)

            data = fifo_rt.read()

            if data:
               current_line = data.strip()

               if ('MEAS_' in current_line):
                  meas_event_time = event_date_time_00.search(current_line)
                  meas_event_datetime = datetime.datetime.strptime(meas_event_time.group(),
                                                                   '%Y-%m-%d  %H:%M:%S')
                  if ('MEAS_START' in current_line):
                     print ("Image recon started at: %s" % str(meas_event_datetime))
                  else: # MEAS_FINISHED
                     print ("Image recon ended at: %s" % str(meas_event_datetime))
               elif ('DICOMIMA' in current_line):
                  # print ('Image file written')
                  pass
               else:
                  print (f'Unknown line received: {current_line}')
            else:
               pass

      return

In [ ]:
   def check_scanner_events_log (scanner_events_dictionary,
                                 events_log='MrMeasContainer.log'):

      """
         This routine will tail the log output from the scanner's generic event log.
      """

      with open (events_log, 'r') as fifo_rt:
         print (f"FIFO '{events_log}' opened for reading. Waiting for data...")

         while True:
            select.select([fifo_rt], [], [], 0.1)

            data = fifo_rt.read()

            if data:
               all_lines = data.strip()

               for current_line in all_lines.splitlines():
                  # print (f'line received: {current_line}')
                  # if True:
                  if ('Starting sequence' in current_line):
                     print (f'starting sequence line received: {current_line}')
                  elif ('MeasCtrlService::handleEvent: Event ' in current_line):
                     print (f'meas control line received: {current_line}')
                  else:
                     pass
            else:
               pass

      return

In [ ]:
if __name__ == "__main__":

   scan_events_dict = dict()

   # check_inline_export_log (scan_events_dict, export_log=FIFO_PATH)
   check_scanner_events_log (scan_events_dict, events_log=EVENT_LOG)